# 02 - OCI consumer to Bronze


In [ ]:
import base64, re
from pathlib import Path
import oci
from pyspark.sql import Row, functions as F

config = {}
for raw in Path('/Workspace/ORACLE_AIDP_STREAMING/config/runtime.yaml').read_text(encoding='utf-8').splitlines():
    line = raw.strip()
    if line and not line.startswith('#') and ':' in line:
        key, value = line.split(':',1)
        config[key.strip()] = value.strip().strip(chr(34)).strip(chr(39))
required = ('OCI_STREAM_ENDPOINT','OCI_STREAM_OCID','OCI_CONFIG_FILE','OCI_PROFILE','AIDP_CATALOG','CONSUMER_GROUP','OCI_STREAM_PARTITIONS')
missing = [key for key in required if not config.get(key)]
if missing: raise ValueError('runtime.yaml is missing: ' + ', '.join(missing))
catalog, group = config['AIDP_CATALOG'], config['CONSUMER_GROUP']
partitions = tuple(value.strip() for value in config['OCI_STREAM_PARTITIONS'].split(',') if value.strip())
if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*',catalog) or not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*',group) or any(not value.isdigit() for value in partitions): raise ValueError('Invalid catalog, consumer group, or partition list')
bronze, checkpoints = f'{catalog}.bronze.meter_reading', f'{catalog}.bronze.stream_consumer_checkpoint'
client = oci.streaming.StreamClient(oci.config.from_file(config['OCI_CONFIG_FILE'],config['OCI_PROFILE']),service_endpoint=config['OCI_STREAM_ENDPOINT'])

for partition in partitions:
    saved = spark.sql(f"SELECT last_offset FROM {checkpoints} WHERE consumer_group='{group}' AND stream_partition='{partition}' ORDER BY committed_at DESC LIMIT 1").collect()
    details = oci.streaming.models.CreateCursorDetails(type='AFTER_OFFSET',offset=saved[0].last_offset) if saved else oci.streaming.models.CreateCursorDetails(type='TRIM_HORIZON')
    cursor = client.create_cursor(config['OCI_STREAM_OCID'],partition,details).data.value
    messages = client.get_messages(config['OCI_STREAM_OCID'],cursor,limit=1000).data.messages
    rows = [Row(stream_partition=partition,stream_offset=str(item.offset),stream_key=base64.b64decode(item.key).decode() if item.key else None,raw_value=base64.b64decode(item.value).decode()) for item in messages]
    if not rows: print(f'No messages for partition {partition}'); continue
    spark.createDataFrame(rows).withColumn('ingested_at',F.current_timestamp()).withColumn('payload_json',F.col('raw_value')).createOrReplaceTempView('bronze_batch')
    spark.sql(f'MERGE INTO {bronze} t USING bronze_batch s ON t.stream_partition=s.stream_partition AND t.stream_offset=s.stream_offset WHEN NOT MATCHED THEN INSERT *')
    spark.createDataFrame([(group,partition,rows[-1].stream_offset)],'consumer_group string, stream_partition string, last_offset string').withColumn('committed_at',F.current_timestamp()).createOrReplaceTempView('checkpoint_batch')
    spark.sql(f'INSERT INTO {checkpoints} SELECT * FROM checkpoint_batch')
    print(f'Processed {len(rows)} messages from partition {partition}')